# 01 · Core DataFrame Operations

Core single-table DataFrame skills: imposing types on raw data,
deriving columns, filtering, and aggregating.

Each task has a **CHECK** cell below it — run it to grade your answer. Write your
solution in the cell with the `# TODO`.

In [ ]:
import sys
from pathlib import Path

_root = Path.cwd()
while not (_root / "utils" / "spark.py").exists() and _root != _root.parent:
    _root = _root.parent
for _p in (str(_root), str(_root / "src")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from utils.spark import build_session, read_dim, read_sales_raw, read_sales_typed

spark = build_session("nb01")
print("Spark", spark.version, "ready")

sales_raw = read_sales_raw(spark)
products = read_dim(spark, "dim_products")
stores = read_dim(spark, "dim_stores")
customers = read_dim(spark, "dim_customers")
print("raw sales rows:", sales_raw.count())
sales_raw.printSchema()

## Task 1: Impose types (safe casting)

The raw sales CSV has **every column as a string**, and `quantity` even contains junk like `"N/A"`. Build a typed DataFrame `sales` from `sales_raw` with `txn_date`→date, `quantity`→int, `unit_price`→double, `discount`→double. Keep all other columns.

Use **safe** casting so bad values become `null` instead of crashing — Spark 4 runs in ANSI mode, so a plain `.cast('int')` on `"N/A"` throws. Use `try_cast` (`F.expr("try_cast(col as int)")`).

In [ ]:
# TODO: build `sales` with proper types using try_cast
sales = sales_raw  # replace me
sales.printSchema()

In [ ]:
# ✅ CHECK — run this cell to grade your answer
types = dict(sales.dtypes)
assert types["txn_date"] == "date", types["txn_date"]
assert types["quantity"] == "int", types["quantity"]
assert types["unit_price"] == "double", types["unit_price"]
assert types["discount"] == "double", types["discount"]
# Casting must keep every column and every row — no dropping/adding.
assert set(sales.columns) == set(sales_raw.columns), set(sales.columns) ^ set(sales_raw.columns)
assert sales.count() == sales_raw.count(), "row count changed during casting"
# Safe cast turned non-numeric quantities ("N/A", ...) into nulls — for exactly the unparseable rows.
_bad = sales_raw.filter(F.expr("try_cast(quantity as int) is null")).count()
assert sales.filter(F.col("quantity").isNull()).count() == _bad, "null quantities != unparseable rows (plain cast?)"
assert _bad > 0, "expected some junk quantities to become null"
print("✅ Task 1 passed —", _bad, "unparseable quantities nulled")

## Task 2: Derive a revenue column

Add a `revenue` column to `sales`, defined as `quantity × unit_price × (1 − discount)`. Name the result `sales_rev`. Rows with a null quantity should naturally get a null revenue.

In [ ]:
# TODO: add `revenue`
sales_rev = sales
sales_rev.select("txn_id", "quantity", "unit_price", "discount").show(5)

In [ ]:
# ✅ CHECK — run this cell to grade your answer
assert "revenue" in sales_rev.columns
assert sales_rev.count() == sales.count()  # withColumn must not change the row count
# Null quantity must propagate to null revenue; revenue is null only where an input is null.
assert sales_rev.filter(F.col("quantity").isNull() & F.col("revenue").isNotNull()).count() == 0, "null qty got non-null revenue"
assert sales_rev.filter(
    F.col("revenue").isNull()
    & F.col("quantity").isNotNull() & F.col("unit_price").isNotNull() & F.col("discount").isNotNull()
).count() == 0, "revenue null despite all inputs present"
# Row-level formula must be exactly quantity * unit_price * (1 - discount).
_wrong = sales_rev.filter(
    F.col("revenue").isNotNull()
    & (F.abs(F.col("revenue") - F.col("quantity") * F.col("unit_price") * (1 - F.col("discount"))) > 1e-9)
).count()
assert _wrong == 0, ("rows with wrong revenue:", _wrong)
print("✅ Task 2 passed — total revenue:", sales_rev.agg(F.round(F.sum("revenue"), 2)).first()[0])

## Task 3: Filter to valid rows

Create `valid_sales` from `sales_rev` keeping only rows with a **positive quantity** AND a **discount within [0, 1]** (this drops negatives, the `N/A` nulls, and out-of-range discounts). Select just: `txn_id`, `txn_date`, `store_id`, `product_id`, `quantity`, `revenue`.

In [ ]:
# TODO
valid_sales = sales_rev

In [ ]:
# ✅ CHECK — run this cell to grade your answer
assert set(valid_sales.columns) == {"txn_id", "txn_date", "store_id", "product_id", "quantity", "revenue"}
# The filter must keep exactly the positive-quantity, in-range-discount rows (recomputed independently).
_expect = sales_rev.filter((F.col("quantity") > 0) & F.col("discount").between(0, 1))
assert valid_sales.count() == _expect.count(), (valid_sales.count(), _expect.count())
assert valid_sales.count() < sales_rev.count(), "nothing was filtered out — check the predicate"
# Invariants on the surviving rows.
assert valid_sales.filter(F.col("quantity").isNull() | (F.col("quantity") <= 0)).count() == 0, "non-positive/null qty survived"
assert valid_sales.filter(F.col("revenue").isNull()).count() == 0, "null revenue survived"
print("✅ Task 3 passed — valid rows:", valid_sales.count())

## Task 4: Aggregate per store

From `valid_sales`, compute per `store_id`: `total_revenue` (rounded to 2 dp), `total_units` (sum of quantity), and `n_txns` (row count). Order by `total_revenue` descending. Name it `store_summary`.

In [ ]:
# TODO
store_summary = valid_sales

In [ ]:
# ✅ CHECK — run this cell to grade your answer
assert set(store_summary.columns) == {"store_id", "total_revenue", "total_units", "n_txns"}
# One row per store, ordered by total_revenue descending.
assert store_summary.count() == valid_sales.select("store_id").distinct().count(), "not one row per store"
_revs = [r["total_revenue"] for r in store_summary.collect()]
assert _revs == sorted(_revs, reverse=True), "rows must be ordered by total_revenue desc"
# Aggregates reconcile with the source totals.
assert store_summary.agg(F.sum("n_txns")).first()[0] == valid_sales.count(), "n_txns doesn't sum to row count"
assert store_summary.agg(F.sum("total_units")).first()[0] == valid_sales.agg(F.sum("quantity")).first()[0], "total_units mismatch"
# Spot-check the top store's numbers against an independent groupBy.
_top = store_summary.first()
_chk = valid_sales.filter(F.col("store_id") == _top["store_id"]).agg(
    F.round(F.sum("revenue"), 2).alias("rev"), F.sum("quantity").alias("u"), F.count("*").alias("n")).first()
assert abs(_top["total_revenue"] - _chk["rev"]) < 0.01, "total_revenue mismatch for top store"
assert _top["total_units"] == _chk["u"] and _top["n_txns"] == _chk["n"], "units/txn mismatch for top store"
assert _top["store_id"] == "ST001", _top["store_id"]  # the mega-store should top the ranking
print("✅ Task 4 passed — top store:", _top["store_id"])

## Task 5: Monthly revenue trend

Add a `year_month` column (formatted `yyyy-MM`) to `valid_sales` and compute monthly total revenue as `monthly_rev` with columns `year_month`, `revenue` (rounded), ordered by `year_month`.

In [ ]:
# TODO
monthly_rev = valid_sales

In [ ]:
# ✅ CHECK — run this cell to grade your answer
import re
assert monthly_rev.columns == ["year_month", "revenue"]
_rows = monthly_rev.collect()
assert len(_rows) >= 6
assert all(re.match(r"^\d{4}-\d{2}$", r["year_month"]) for r in _rows), "year_month not formatted yyyy-MM"
# One row per month, ordered ascending.
_months = [r["year_month"] for r in _rows]
assert _months == sorted(_months), "rows must be ordered by year_month"
assert len(_months) == len(set(_months)), "months must be distinct (proper groupBy)"
# Monthly totals must reconcile with the overall valid revenue.
_sum = sum(r["revenue"] for r in _rows)
_tot = valid_sales.agg(F.sum("revenue")).first()[0]
assert abs(_sum - _tot) < 1.0, ("monthly sum != total", _sum, _tot)
print("✅ Task 5 passed —", len(_months), "months")

---
🎉 Finished! Re-run the notebook top-to-bottom; every CHECK cell should print a ✅.